# Conformal TB Triage — GPU Sensitivity Analyses (held-out conformal calibration)

**Runtime: GPU (A100/V100/T4).** Supersedes the earlier version of this notebook,
which computed the conformal calibration scores on the **same** NLM `calibration`
split used to train the probe — the in-sample-calibration defect addressed in the
held-out calibration revision.

**Held-out design (matches `conformal_pipeline.py`):**
- Probe: linear (raw embeddings, CV-selected `C`) trained on NLM `calibration`.
- Conformal calibration: **held-out TBX11K `dev`** split (out-of-sample for the
  probe), 50/50 stratified (SEED=42); the conformal half sets **Mondrian**
  thresholds on **raw** scores at **α=0.10** (manuscript Table 3 primary config).
- Only the **test** images are re-extracted under each manipulation; the
  calibration thresholds are derived once from the *saved* `dev` predictions, so
  they are exact and GPU-independent.

**Two hard reproduction gates (do not trust results if either fails):**
- **Sanity A (CPU):** retrained probe must reproduce the saved `test` predictions
  to < 1e-3 — guards the probe recipe (raw, not L2-normalised; CV-selected `C`).
- **Sanity B (GPU):** re-extracted clean embeddings must match the stored ones —
  guards against `transformers` version drift in embedding extraction.

**Analyses:** §5.4.2 image-quality degradation · §5.4.3 TTA (K=5) ·
§5.4.17 lung segmentation (whole / lung-only / mediastinal-only) ·
§5.5.15 prediction-set stability (K=10). All report **marginal** coverage,
TB-class coverage, and **empty-set** fraction — the metrics that exposed the
in-sample defect. Outputs are written under the canonical held-out names; regenerate
the image-degradation supplementary figure and the supplement text from them.

In [ ]:
# ── Cell 1: Setup — environment, pinned deps, inputs, RAD-DINO ────────
import subprocess, sys, os, time
from pathlib import Path

IN_COLAB = ("google.colab" in sys.modules) or os.path.exists("/content")
SEED = 42

# Pin ONLY the libraries that affect embedding reproducibility (transformers/timm).
# We deliberately do NOT pin numpy/pandas/pyarrow/scikit-learn: Colab's base image
# ships a mutually ABI-consistent NumPy-2 scientific stack, and force-installing
# older NumPy-1.x-compiled wheels (pandas==2.2.2 etc.) breaks with
# "ImportError: cannot import name '_center' from 'numpy._core.umath'". torch is
# also left to the host CUDA build. Reproducibility is enforced empirically by
# Sanity A (probe, Cell 2) and Sanity B (embeddings, Cell 4), not by exact pins.
PINS = ["transformers==4.44.2", "timm==1.0.9"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=False)
# lungmask (optional, segmentation only) is installed lazily in Cell 6 so it can
# never downgrade numpy out from under the core stack used by Cells 2-5/7.

import numpy as np, pandas as pd, torch, gc
import torchvision.transforms.functional as TF
from PIL import Image, ImageEnhance
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score

np.random.seed(SEED); torch.manual_seed(SEED)
print(f"numpy={np.__version__}  pandas={pd.__version__}  torch={torch.__version__}", flush=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} ({p.total_memory/1e9:.1f} GB)", flush=True)
else:
    print("WARNING: no GPU detected — re-extraction will be very slow. "
          "Runtime > Change runtime type > GPU.", flush=True)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/conformal-tb-triage")
else:
    DRIVE_ROOT = Path(os.environ.get("TB_TRIAGE_ROOT", "."))

EMB_DIR     = DRIVE_ROOT / "data" / "interim" / "embeddings"
PROC        = DRIVE_ROOT / "data" / "processed"
RESULTS_DIR = DRIVE_ROOT / "results" / "tables"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

required = {
    "splits":      PROC / "splits.parquet",
    "rad_dino":    EMB_DIR / "rad_dino.parquet",
    "saved_preds": DRIVE_ROOT / "results" / "probe_predictions.parquet",
}
missing = {k: str(v) for k, v in required.items() if not v.exists()}
assert not missing, f"Missing required inputs on Drive: {missing}"

splits     = pd.read_parquet(required["splits"])
baseline   = pd.read_parquet(required["rad_dino"])
saved_pred = pd.read_parquet(required["saved_preds"])
emb_cols   = [c for c in baseline.columns if c.startswith("emb_")]
print(f"splits={len(splits)}  rad_dino_emb={len(baseline)}  emb_dim={len(emb_cols)}  "
      f"saved_pred={len(saved_pred)}", flush=True)

from transformers import AutoModel, AutoImageProcessor
print("Loading RAD-DINO (microsoft/rad-dino)...", flush=True)
processor = AutoImageProcessor.from_pretrained("microsoft/rad-dino", use_fast=False)
model = AutoModel.from_pretrained("microsoft/rad-dino").to(device).eval()
print("Ready.", flush=True)

In [ ]:
# ── Cell 2: Held-out conformal pipeline ──────────────────────────────
# Thresholds come from the HELD-OUT dev split (raw scores, Mondrian, alpha=0.10);
# the probe is retrained with the real recipe so it can score re-extracted images.
ALPHA = 0.10
EMB, PROBE = "rad_dino", "linear"

def mondrian_thresholds(cal_prob, cal_y, alpha=ALPHA):
    s = np.where(cal_y == 1, 1 - cal_prob, cal_prob)
    thr = {}
    for c in (0, 1):
        sc = s[cal_y == c]; n = len(sc)
        thr[c] = 1.0 if n < 5 else float(np.quantile(sc, min(np.ceil((n + 1) * (1 - alpha)) / n, 1.0)))
    return thr

def eval_sets(test_prob, y, thr):
    inc1 = (1 - test_prob) <= thr.get(1, 1.0)
    inc0 = test_prob <= thr.get(0, 1.0)
    covered = np.where(y == 1, inc1, inc0)
    size = inc0.astype(int) + inc1.astype(int)
    tb, ntb = y == 1, y == 0
    tb_cov  = float(covered[tb].mean())  if tb.any()  else float("nan")
    ntb_cov = float(covered[ntb].mean()) if ntb.any() else float("nan")
    auroc = roc_auc_score(y, test_prob) if len(np.unique(y)) > 1 else float("nan")
    return {"auroc": round(float(auroc), 4),
            "marginal_cov": round(float(covered.mean()), 4),
            "tb_cov": round(tb_cov, 4), "nontb_cov": round(ntb_cov, 4),
            "mean_size": round(float(size.mean()), 4),
            "singleton": round(float((size == 1).mean()), 4),
            "empty": round(float((size == 0).mean()), 4),
            "disparity": round(abs(tb_cov - ntb_cov), 4)}

# (a) FIXED thresholds from SAVED held-out dev predictions (exact; no GPU).
sp = saved_pred[(saved_pred.embedding == EMB) & (saved_pred.probe == PROBE)]
dev = sp[sp.split == "dev"]
dev_p, dev_y = dev.y_prob.to_numpy(), dev.y_true.to_numpy().astype(int)
_, conf_idx = train_test_split(np.arange(len(dev_p)), test_size=0.5,
                               random_state=SEED, stratify=dev_y)
THR = mondrian_thresholds(dev_p[conf_idx], dev_y[conf_idx])
print(f"Fixed Mondrian thresholds (held-out dev conf half, n={len(conf_idx)}): "
      f"thr0={THR[0]:.4f} thr1={THR[1]:.4f}", flush=True)

# (b) Retrain the probe with the REAL recipe: raw embeddings (NO L2-norm),
#     CV-selected C (NOT fixed C=10). Matches src/probes/train_probes.py.
def get_xy(df, split_name):
    m = (df["split"] == split_name) & (df["tb_binary"].isin(["tb_positive", "tb_negative"]))
    s = df[m]
    return (s[emb_cols].values.astype(np.float32),
            (s["tb_binary"] == "tb_positive").astype(int).values,
            s["patient_id"].values)

def train_linear_probe(Xc, yc):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    best_c, best = None, -1.0
    for C in [1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100]:
        fold = []
        for tr, va in skf.split(Xc, yc):
            m = LogisticRegression(C=C, max_iter=2000, solver="lbfgs",
                                   random_state=SEED).fit(Xc[tr], yc[tr])
            if len(np.unique(yc[va])) > 1:
                fold.append(roc_auc_score(yc[va], m.predict_proba(Xc[va])[:, 1]))
        if fold and np.mean(fold) > best:
            best, best_c = float(np.mean(fold)), C
    final = LogisticRegression(C=best_c, max_iter=2000, solver="lbfgs",
                               random_state=SEED).fit(Xc, yc)
    return final, best_c

X_cal, y_cal, _ = get_xy(baseline, "calibration")
probe, best_c = train_linear_probe(X_cal, y_cal)
print(f"Retrained linear probe (CV-selected C={best_c}).", flush=True)

def score(emb_matrix):
    return probe.predict_proba(emb_matrix.astype(np.float32))[:, 1]

# (c) SANITY A (CPU): retrained probe reproduces saved test predictions.
X_ts, y_ts, pid_ts = get_xy(baseline, "test")
repro = score(X_ts)
ref = sp[sp.split == "test"].set_index("patient_id").loc[pid_ts, "y_prob"].to_numpy()
maxd = float(np.max(np.abs(repro - ref)))
print(f"SANITY A (probe reproduction): max|d prob| = {maxd:.2e}", flush=True)
assert maxd < 1e-3, (f"Retrained probe does not reproduce saved predictions "
                     f"(max|d|={maxd:.2e}); check scikit-learn pin.")

# (d) Anchor headline on clean saved test predictions (expect marginal ~0.914).
anchor = eval_sets(ref, y_ts, THR)
print(f"Held-out headline (clean test): marginal={anchor['marginal_cov']:.4f} "
      f"TB={anchor['tb_cov']:.4f} empty={anchor['empty']:.4f}", flush=True)

In [ ]:
# ── Cell 3: Cache TEST images locally + STREAMING extraction helper ──
# We do NOT load the images into RAM. CXRs are large (many >2000px); holding all
# 5,879 decoded at once exhausts the T4 box's ~13 GB (the earlier OOM). Instead we
# keep the local file paths and stream small batches off disk on demand.
import shutil
LOCAL = Path("/content/local_images") if IN_COLAB else (DRIVE_ROOT / "_local_cache")
LOCAL.mkdir(parents=True, exist_ok=True)

test_meta = splits[(splits.split == "test") &
                   (splits.tb_binary.isin(["tb_positive", "tb_negative"]))].copy()
print(f"Test images: {len(test_meta)}", flush=True)

def _localise(fp):
    src = Path(fp); dst = LOCAL / src.name
    if not dst.exists():
        try: shutil.copy(src, dst)
        except Exception: return str(src)
    return str(dst)

if IN_COLAB:
    tqdm.pandas(desc="Caching")
    test_meta["local_path"] = test_meta["file_path"].progress_apply(_localise)
else:
    test_meta["local_path"] = test_meta["file_path"]

# Verify each image opens WITHOUT decoding/holding it (verify() is cheap).
ok = []
for p in tqdm(test_meta["local_path"].tolist(), desc="Verifying"):
    try:
        with Image.open(p) as im: im.verify()
        ok.append(True)
    except Exception:
        ok.append(False)
ok = np.array(ok, dtype=bool)
test_meta = test_meta[ok].reset_index(drop=True)

TEST_PATHS = test_meta["local_path"].tolist()
YV   = (test_meta["tb_binary"] == "tb_positive").astype(int).to_numpy()
PIDV = test_meta["patient_id"].to_numpy()
cached = int(test_meta["local_path"].str.startswith(str(LOCAL)).sum())
print(f"Usable test images: {len(TEST_PATHS)} (cached locally: {cached})", flush=True)

# Streaming extractor: opens images from disk in small batches, applies an optional
# per-image transform (a degradation/augmentation), extracts CLS embeddings, frees
# the batch. Peak RAM is bounded by batch_size, NOT the full corpus.
def extract_paths(paths, transform=None, batch_size=32, desc="extract"):
    out = []
    for i in tqdm(range(0, len(paths), batch_size), desc=desc, leave=False):
        imgs = []
        for p in paths[i:i + batch_size]:
            im = Image.open(p).convert("RGB")
            imgs.append(transform(im) if transform is not None else im)
        px = processor(images=imgs, return_tensors="pt")["pixel_values"].to(device)
        with torch.no_grad():
            e = model(pixel_values=px).last_hidden_state[:, 0, :]
        out.append(e.cpu().float().numpy())
        del imgs, px, e
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return np.vstack(out) if out else np.empty((0, len(emb_cols)), dtype=np.float32)

In [ ]:
# ── Cell 4: §5.4.2 IMAGE QUALITY DEGRADATION (held-out, streaming) ───
import io
OUT = RESULTS_DIR / "image_degradation.csv"

# SANITY B (GPU): re-extracted clean embeddings reproduce stored ones.
chk_n = min(200, len(TEST_PATHS))
chk_emb = extract_paths(TEST_PATHS[:chk_n], desc="sanityB")
stored = baseline.set_index("patient_id").loc[PIDV[:chk_n], emb_cols].to_numpy()
bdiff = float(np.max(np.abs(chk_emb - stored)))
print(f"SANITY B (embedding reproduction, n={chk_n}): max|d emb| = {bdiff:.2e}", flush=True)
if bdiff > 1e-2:
    print("  WARNING: re-extracted embeddings differ from stored (>1e-2). Pin "
          "transformers to the extraction-time version before trusting results.", flush=True)

# Resume: load any rows already computed and skip them (crash-safe).
if OUT.exists():
    rows = pd.read_csv(OUT).to_dict("records")
    DONE = {(r["degradation"], r["level"]) for r in rows}
    print(f"Resuming: {len(DONE)} degradation row(s) already present.", flush=True)
else:
    rows, DONE = [], set()

def run_deg(degradation, level, transform):
    if (degradation, level) in DONE:
        print(f"  skip {degradation}/{level} (done)", flush=True); return
    emb = extract_paths(TEST_PATHS, transform, desc=f"{degradation}/{level}")
    rows.append({"degradation": degradation, "level": level,
                 **eval_sets(score(emb), YV, THR)})
    pd.DataFrame(rows).to_csv(OUT, index=False)   # incremental save after each row
    r = rows[-1]
    print(f"  {degradation}/{level}: marginal={r['marginal_cov']:.4f} "
          f"TB={r['tb_cov']:.4f} empty={r['empty']:.4f}", flush=True)

run_deg("none", "original", None)
clean = next((r for r in rows if r["degradation"] == "none"), None)
if clean and abs(clean["marginal_cov"] - anchor["marginal_cov"]) > 0.02:
    print(f"  WARNING: re-extracted clean marginal {clean['marginal_cov']:.3f} "
          f"deviates from saved anchor {anchor['marginal_cov']:.3f}.", flush=True)

for res in [128, 64, 32]:
    run_deg("resolution", f"{res}x{res}",
            lambda im, r=res: im.resize((r, r), Image.BILINEAR).resize(im.size, Image.BILINEAR))

def jpeg_tf(im, q):
    buf = io.BytesIO(); im.save(buf, format="JPEG", quality=q); buf.seek(0)
    return Image.open(buf).convert("RGB")
for q in [50, 30, 10]:
    run_deg("jpeg", f"q{q}", lambda im, q=q: jpeg_tf(im, q))

def noise_tf(im, sigma):
    rng = np.random.default_rng(SEED)
    a = np.asarray(im).astype(np.float32) / 255.0
    a = np.clip(a + rng.normal(0, sigma, a.shape), 0, 1)
    return Image.fromarray((a * 255).astype(np.uint8))
for sigma in [0.01, 0.05, 0.10]:
    run_deg("noise", f"sigma{sigma}", lambda im, s=sigma: noise_tf(im, s))

for name, fac in [("bright+20%", 1.2), ("bright-20%", 0.8),
                  ("bright+40%", 1.4), ("bright-40%", 0.6)]:
    run_deg("brightness", name, lambda im, f=fac: ImageEnhance.Brightness(im).enhance(f))

print("Saved image_degradation.csv")
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# ── Cell 5: §5.4.3 TEST-TIME AUGMENTATION (K=5, streaming) ──────────
OUT = RESULTS_DIR / "tta_results.csv"
if OUT.exists():
    print("tta_results.csv exists — skipping.")
    print(pd.read_csv(OUT).to_string(index=False))
else:
    augs = [("hflip",     lambda im: TF.hflip(im)),
            ("rot+5",     lambda im: TF.rotate(im, 5, fill=0)),
            ("rot-5",     lambda im: TF.rotate(im, -5, fill=0)),
            ("bright+10", lambda im: ImageEnhance.Brightness(im).enhance(1.1)),
            ("bright-10", lambda im: ImageEnhance.Brightness(im).enhance(0.9))]
    emb_orig = extract_paths(TEST_PATHS, desc="tta/orig")
    emb_sum = np.zeros_like(emb_orig)
    for name, fn in augs:
        emb_sum += extract_paths(TEST_PATHS, fn, desc=f"tta/{name}")
    emb_tta = emb_sum / len(augs)
    tta_df = pd.DataFrame([{"method": "no_tta", **eval_sets(score(emb_orig), YV, THR)},
                           {"method": "tta_k5", **eval_sets(score(emb_tta), YV, THR)}])
    tta_df.to_csv(OUT, index=False)
    print("Saved tta_results.csv"); print(tta_df.to_string(index=False))
    del emb_orig, emb_sum, emb_tta

In [ ]:
# ── Cell 6: §5.4.17 LUNG SEGMENTATION (whole / lung-only / mediastinal) ─
OUT = RESULTS_DIR / "lung_segmentation.csv"
if OUT.exists():
    print("lung_segmentation.csv exists — skipping.")
    print(pd.read_csv(OUT).to_string(index=False))
else:
    # Lazy install here (kept out of Cell 1 so it cannot downgrade the core stack).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "lungmask", "SimpleITK"], check=False)
    try:
        from lungmask import LMInferer; import SimpleITK as sitk
        inferer = LMInferer(modelname="R231"); USE_LM = True
        print("Using lungmask R231.", flush=True)
    except Exception as e:
        USE_LM = False
        print(f"lungmask unavailable ({e}); intensity-based lung approximation.", flush=True)

    n_seg = min(1000, len(TEST_PATHS))
    rng = np.random.default_rng(SEED)
    sel = rng.choice(len(TEST_PATHS), n_seg, replace=False)
    sel_paths = [TEST_PATHS[i] for i in sel]; ys = YV[sel]

    def lung_mask(im):
        if USE_LM:
            try:
                a = np.asarray(im.convert("L"))
                m = inferer.apply(sitk.GetImageFromArray(a[None]))[0] > 0
                return m.astype(bool)
            except Exception:
                pass
        a = np.asarray(im.convert("L")).astype(float)
        lo, hi = np.percentile(a, 20), np.percentile(a, 80)
        return (a > lo) & (a < hi)

    def mk_whole(im): return im
    def mk_lung(im):
        m = lung_mask(im); a = np.asarray(im).copy(); a[~m] = 0; return Image.fromarray(a)
    def mk_medi(im):
        m = lung_mask(im); a = np.asarray(im).copy(); a[m] = 0;  return Image.fromarray(a)

    # Stream each condition independently so RAM stays bounded by batch_size.
    res = {"whole_image":      eval_sets(score(extract_paths(sel_paths, mk_whole, desc="seg/whole")), ys, THR),
           "lung_only":        eval_sets(score(extract_paths(sel_paths, mk_lung,  desc="seg/lung")),  ys, THR),
           "mediastinal_only": eval_sets(score(extract_paths(sel_paths, mk_medi,  desc="seg/medi")),  ys, THR)}
    seg_df = pd.DataFrame([{"condition": k, "n": n_seg, "lungmask": USE_LM, **v}
                           for k, v in res.items()])
    seg_df.to_csv(OUT, index=False)
    print("Saved lung_segmentation.csv"); print(seg_df.to_string(index=False))

In [ ]:
# ── Cell 7: §5.5.15 PREDICTION-SET STABILITY (K=10) ─────────────────
OUT = RESULTS_DIR / "perturbation_stability.csv"
if OUT.exists():
    print("perturbation_stability.csv exists — skipping.")
    print(pd.read_csv(OUT).head().to_string(index=False))
else:
    n_stab, K = 200, 10
    rng = np.random.default_rng(SEED)
    sel = rng.choice(len(TEST_PATHS), n_stab, replace=False)
    sel_paths = [TEST_PATHS[i] for i in sel]; ys = YV[sel]

    def to_set(p):
        s = set()
        if (1 - p) <= THR.get(1, 1.0): s.add(1)
        if p <= THR.get(0, 1.0):       s.add(0)
        return frozenset(s)

    def perturb(im):
        w, h = im.size; cf = rng.uniform(0.98, 1.0)
        q = TF.center_crop(im, [int(h * cf), int(w * cf)]).resize((w, h), Image.BILINEAR)
        a = np.asarray(q).astype(np.float32) / 255.0
        a = np.clip(a + rng.normal(0, 0.01, a.shape), 0, 1)
        return Image.fromarray((a * 255).astype(np.uint8))

    orig_sets = [to_set(p) for p in score(extract_paths(sel_paths, desc="stab/orig"))]
    match = np.zeros((n_stab, K))
    for k in range(K):
        ps = [to_set(p) for p in score(extract_paths(sel_paths, perturb, desc=f"stab/k{k}"))]
        match[:, k] = [o == p for o, p in zip(orig_sets, ps)]
    stab = match.mean(axis=1)
    print(f"Mean set-stability {stab.mean():.4f}; <0.80 in {(stab < 0.80).mean():.1%} of images",
          flush=True)
    df = pd.DataFrame({"y_true": ys, "orig_set": [str(set(s)) for s in orig_sets],
                       "stability": stab})
    df.to_csv(OUT, index=False)
    summ = df.groupby("orig_set")["stability"].agg(["count", "mean"]).reset_index()
    summ.to_csv(RESULTS_DIR / "perturbation_stability_summary.csv", index=False)
    print(summ.to_string(index=False)); print("Saved perturbation_stability.csv")

In [ ]:
# ── Cell 8: Verify outputs ──────────────────────────────────────────
print("Held-out GPU sensitivity analyses — outputs on Drive (results/tables/):")
for f in ["image_degradation.csv", "tta_results.csv",
          "lung_segmentation.csv", "perturbation_stability.csv",
          "perturbation_stability_summary.csv"]:
    p = RESULTS_DIR / f
    print(f"  {f:48s} {('%.1f KB' % (p.stat().st_size/1e3)) if p.exists() else 'MISSING'}")
print("\nAll use the held-out conformal calibration (raw RAD-DINO/linear, "
      "Mondrian alpha=0.10). Download to results/tables/, then regenerate the image-degradation supplementary figure "
      "and reconcile the supplement methods grid (S4 + line 23) against these values.")
del model, processor; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()